In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb
from scipy.special import softmax
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns",None)

from sklearn.linear_model import LogisticRegression

In [2]:
train = pd.read_csv('/kaggle/input/multi-class-prediction-of-cirrhosis-outcomess/train.csv')
test = pd.read_csv('/kaggle/input/multi-class-prediction-of-cirrhosis-outcomess/test.csv')
sample_sub = pd.read_csv('/kaggle/input/multi-class-prediction-of-cirrhosis-outcomess/sample_submission.csv')
train.drop(columns=["id"],axis=1,inplace=True)

In [3]:
print("Check Out Train DaTA Null Values: ",train.isnull().sum())
print("#"*130)
print(f"Train Data Shape: {train.shape}")
print("#"*130)
print(f"Train Data INFO: {train.info()}")
print("#"*130)

Check Out Train DaTA Null Values:  N_Days           0
Drug             0
Age              0
Sex              0
Ascites          0
Hepatomegaly     0
Spiders          0
Edema            0
Bilirubin        0
Cholesterol      0
Albumin          0
Copper           0
Alk_Phos         0
SGOT             0
Tryglicerides    0
Platelets        0
Prothrombin      0
Stage            0
Status           0
dtype: int64
##################################################################################################################################
Train Data Shape: (7905, 19)
##################################################################################################################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7905 entries, 0 to 7904
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   N_Days         7905 non-null   int64  
 1   Drug           7905 non-null   object 
 2   Age         

In [4]:
print("Check Out Test DaTA Null Values: ",test.isnull().sum())
print("#"*130)
print(f"Test Data Shape: {test.shape}")
print("#"*130)
print(f"Test Data INFO: {test.info()}")
print("#"*130)

Check Out Test DaTA Null Values:  id               0
N_Days           0
Drug             0
Age              0
Sex              0
Ascites          0
Hepatomegaly     0
Spiders          0
Edema            0
Bilirubin        0
Cholesterol      0
Albumin          0
Copper           0
Alk_Phos         0
SGOT             0
Tryglicerides    0
Platelets        0
Prothrombin      0
Stage            0
dtype: int64
##################################################################################################################################
Test Data Shape: (5271, 19)
##################################################################################################################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5271 entries, 0 to 5270
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             5271 non-null   int64  
 1   N_Days         5271 non-null   int64  
 2   Drug          

In [5]:
train.head()

,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,999,D-penicillamine,21532,M,N,N,N,N,2.3,316.0,3.35,172.0,1601.0,179.80,63.0,394.0,9.7,3.0,D
1,2574,Placebo,19237,F,N,N,N,N,0.9,364.0,3.54,63.0,1440.0,134.85,88.0,361.0,11.0,3.0,C
2,3428,Placebo,13727,F,N,Y,Y,Y,3.3,299.0,3.55,131.0,1029.0,119.35,50.0,199.0,11.7,4.0,D
3,2576,Placebo,18460,F,N,N,N,N,0.6,256.0,3.50,58.0,1653.0,71.30,96.0,269.0,10.7,3.0,C
4,788,Placebo,16658,F,N,Y,N,N,1.1,346.0,3.65,63.0,1181.0,125.55,96.0,298.0,10.6,4.0,C


In [6]:
test.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,7905,3839,D-penicillamine,19724,F,N,Y,N,N,1.2,546.0,3.37,65.0,1636.0,151.90,90.0,430.0,10.6,2.0
1,7906,2468,D-penicillamine,14975,F,N,N,N,N,1.1,660.0,4.22,94.0,1257.0,151.90,155.0,227.0,10.0,2.0
2,7907,51,Placebo,13149,F,N,Y,N,Y,2.0,151.0,2.96,46.0,961.0,69.75,101.0,213.0,13.0,4.0
3,7908,2330,D-penicillamine,20510,F,N,N,N,N,0.6,293.0,3.85,40.0,554.0,125.55,56.0,270.0,10.6,2.0
4,7909,1615,D-penicillamine,21904,F,N,Y,N,N,1.4,277.0,2.97,121.0,1110.0,125.00,126.0,221.0,9.8,1.0


In [7]:
def feature_engineer(df):
    for c in ['Bilirubin','Cholesterol','Copper','Alk_Phos','SGOT','Tryglicerides','Prothrombin']:
        df[f'log_{c}'] = np.log1p(df[c])
    df['Age'] = df['Age'] // 365
    df['is_male'] = (df['Sex'] == 'M').astype(int)
    df['edema_score'] = df['Edema'].map({'N':0, 'S':0.5, 'Y':1})
    df['risk'] = df['Bilirubin'] * df['Prothrombin']
    df['liver_health'] = df['Albumin'] / (df['log_Bilirubin'] + 0.1)
    bool_cols = ['Ascites','Hepatomegaly','Spiders']
    for c in bool_cols:
        df[c] = df[c].map({'N':0, 'Y':1})
    return df

train = feature_engineer(train)
test = feature_engineer(test)


In [8]:
train["Status"]=train['Status'].map({'C':0, 'CL':1, 'D':2})

In [9]:
cat_cols=["Drug","Sex","Edema"]

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])


X = train.drop(['Status'], axis=1)
y = train['Status']
X_test = test.drop('id', axis=1)

In [10]:
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

lgb_oof = np.zeros((len(X), 3))
xgb_oof = np.zeros((len(X), 3))
lgb_preds = np.zeros((len(X_test), 3))
xgb_preds = np.zeros((len(X_test), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    lgb_model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=3,
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=8,
        num_leaves=70,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        min_child_samples=20,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100)])
    lgb_oof[val_idx] = lgb_model.predict_proba(X_val)
    lgb_preds += lgb_model.predict_proba(X_test) / n_splits
    
    xgb_model = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        min_child_weight=5,
        random_state=42 + fold,
        n_jobs=-1,
        verbosity=0
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=False)
    xgb_oof[val_idx] = xgb_model.predict_proba(X_val)
    xgb_preds += xgb_model.predict_proba(X_test) / n_splits

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[67]	valid_0's multi_logloss: 0.452731
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[73]	valid_0's multi_logloss: 0.454933
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[81]	valid_0's multi_logloss: 0.455307
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[74]	valid_0's multi_logloss: 0.447615
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[76]	valid_0's multi_logloss: 0.407771
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[104]	valid_0's multi_logloss: 0.436904
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[67]	valid_0's multi_logloss: 0.445157
Training until validation scores don't improve 

In [11]:
meta_X = np.hstack([lgb_oof, xgb_oof])
meta = LogisticRegression(multi_class='multinomial', max_iter=2000)
meta.fit(meta_X, y)

w1 = max(meta.coef_[0][:3].mean(), 0)
w2 = max(meta.coef_[0][3:].mean(), 0)
total = w1 + w2
w_lgb = w1 / total if total > 0 else 0.5
w_xgb = w2 / total if total > 0 else 0.5

final_preds = w_lgb * lgb_preds + w_xgb * xgb_preds
final_preds = np.clip(final_preds, 1e-15, 1-1e-15)
final_preds = final_preds / final_preds.sum(axis=1, keepdims=True)

final_oof = w_lgb * lgb_oof + w_xgb * xgb_oof
print('Final blended CV:', log_loss(y, final_oof))
print(f'Blend → LGBM {w_lgb:.3f} | XGBoost {w_xgb:.3f}')

Final blended CV: 0.44486462522650566
Blend → LGBM 0.500 | XGBoost 0.500


In [12]:
sub = sample_sub.copy()
sub[['Status_C','Status_CL','Status_D']] = final_preds
sub.to_csv('submission.csv', index=False)
sub.head()

,id,Status_C,Status_CL,Status_D
0,7905,0.648737,0.035737,0.315525
1,7906,0.741293,0.146277,0.112430
2,7907,0.020050,0.010457,0.969493
3,7908,0.947205,0.005656,0.047139
4,7909,0.831132,0.033142,0.135727
